# Divisão Treino / Validação / Teste


In [1]:
import os

import pandas as pd

ROOT_PATH = os.path.dirname(os.getcwd())
DATA_FOLDER_PATH = os.path.join(ROOT_PATH, 'data', 'processed')

In [2]:
df = pd.read_parquet(os.path.join(DATA_FOLDER_PATH, 'cleaned_arrivals_ptbr.parquet'))

df.head()


,access_route,state,origin_country,month,year,arrival_count,month_num,date,year_month,state_region,country_cultural_subdivision
0,Aérea,Bahia,Afeganistão,Janeiro,2026,1.0,01,2026-01-31,2026-01-31,Região Nordeste,MENA
1,Aérea,São Paulo,Afeganistão,Janeiro,2026,4.0,01,2026-01-31,2026-01-31,Região Sudeste,MENA
2,Terrestre,Paraná,Afeganistão,Janeiro,2026,1.0,01,2026-01-31,2026-01-31,Região Sul,MENA
3,Terrestre,Rio Grande do Sul,Afeganistão,Janeiro,2026,1.0,01,2026-01-31,2026-01-31,Região Sul,MENA
4,Marítima,Rio de Janeiro,Afeganistão,Janeiro,2026,3.0,01,2026-01-31,2026-01-31,Região Sudeste,MENA


## Divisão por Tempo

A série mensal é dividida em três blocos consecutivos:

| Divisão | Meses | Usado para |
|---|---|---|
| **treino** (`train`) | tudo antes do ano de validação | ajuste (treinamento) dos modelos |
| **validação** (`validation`) | os `VAL_MONTHS` = 12 meses anteriores ao ano de teste | comparação e seleção de modelos (notebooks 5 e 7) |
| **teste** (`test`) | os **últimos `TEST_MONTHS` = 12 meses** dos dados | avaliação final única, realizada no notebook final |

O conjunto de teste é mantido separado até o fim: ele só é lido uma única vez, pelo notebook final, após
o modelo ter sido escolhido no ano de validação. A EDA (notebooks 3 e 4) e a engenharia de features
(notebook 6) têm acesso apenas a treino + validação, nunca aos meses de teste.

As datas de divisão são **derivadas dos dados**, nunca digitadas manualmente: quando um novo CSV é adicionado
e os notebooks são reexecutados, os três blocos avançam automaticamente. Todos os notebooks posteriores
leem suas datas a partir de `train_ptbr.parquet`, `val_ptbr.parquet` e `test_ptbr.parquet`, portanto esta
célula é o único local que as define.

Duas verificações protegem a divisão:

- **Sem meses ausentes.** A série mensal agregada deve ser contígua do primeiro ao
  último mês. Uma lacuna deslocaria as janelas de 12 meses.
- **Último mês completo.** O ministério às vezes publica meses parciais. Se o último mês
  estiver abaixo de 50% do mesmo mês um ano antes, provavelmente está incompleto, e um aviso
  é emitido. Remova as linhas desse mês (ou aguarde a publicação completa) antes de modelar.

In [3]:
VAL_MONTHS = 12
TEST_MONTHS = 12
INCOMPLETE_MONTH_RATIO = 0.5

monthly_totals = df.groupby('date')['arrival_count'].sum().sort_index()

expected_months = pd.date_range(
    monthly_totals.index.min(), monthly_totals.index.max(), freq='ME'
)
missing_months = expected_months.difference(monthly_totals.index)
assert missing_months.empty, f'Meses ausentes nos dados: {list(missing_months)}'

last_month = monthly_totals.index.max()
test_start = last_month - pd.offsets.MonthEnd(TEST_MONTHS - 1)
val_start = test_start - pd.offsets.MonthEnd(VAL_MONTHS)

print(
    f'Período completo: {monthly_totals.index.min():%B %Y} a {last_month:%B %Y} '
    f'({len(monthly_totals)} meses)'
)
val_end = test_start - pd.offsets.MonthEnd(1)
print(f'Janela de validação: {val_start:%B %Y} a {val_end:%B %Y}')
print(f'Janela de teste    : {test_start:%B %Y} a {last_month:%B %Y}')

same_month_last_year = last_month - pd.offsets.MonthEnd(12)
ratio = monthly_totals[last_month] / monthly_totals[same_month_last_year]
print(
    f'Último mês vs mesmo mês um ano antes: {ratio:.2f} '
    f'({monthly_totals[last_month]:,.0f} vs '
    f'{monthly_totals[same_month_last_year]:,.0f})'
)
if ratio < INCOMPLETE_MONTH_RATIO:
    print(
        f'AVISO: {last_month:%B %Y} parece incompleto. Remova-o do CSV bruto '
        'ou aguarde o release completo antes de modelar.'
    )


Período completo: January 1989 a August 2026 (452 meses)
Janela de validação: September 2024 a August 2025
Janela de teste    : September 2025 a August 2026
Último mês vs mesmo mês um ano antes: 1.00 (576,276 vs 576,049)


In [4]:
train_df = df[df['date'] < val_start]
print(
    f'Período do conjunto de treino: {train_df["date"].min():%d %B %Y} a '
    f'{train_df["date"].max():%d %B %Y}'
)
train_df.tail()


Período do conjunto de treino: 31 January 1989 a 31 August 2024


,access_route,state,origin_country,month,year,arrival_count,month_num,date,year_month,state_region,country_cultural_subdivision
952675,Fluvial,Rio Grande do Sul,Holanda,Agosto,2024,1.0,08,2024-08-31,2024-08-31,Região Sul,Western / Eastern Europe
952676,Fluvial,Rio Grande do Sul,Itália,Agosto,2024,1.0,08,2024-08-31,2024-08-31,Região Sul,Western / Eastern Europe
952677,Fluvial,Rio Grande do Sul,Paraguai,Agosto,2024,102.0,08,2024-08-31,2024-08-31,Região Sul,LATAM
952678,Fluvial,Rio Grande do Sul,Rússia,Agosto,2024,3.0,08,2024-08-31,2024-08-31,Região Sul,Western / Eastern Europe
952679,Fluvial,Rio Grande do Sul,Uruguai,Agosto,2024,2.0,08,2024-08-31,2024-08-31,Região Sul,LATAM


In [5]:
val_df = df[(df['date'] >= val_start) & (df['date'] < test_start)]
assert val_df['date'].nunique() == VAL_MONTHS
print(
    f'Período do conjunto de validação: {val_df["date"].min():%d %B %Y} a '
    f'{val_df["date"].max():%d %B %Y}'
)
val_df.tail()


Período do conjunto de validação: 30 September 2024 a 31 August 2025


,access_route,state,origin_country,month,year,arrival_count,month_num,date,year_month,state_region,country_cultural_subdivision
976740,Aérea,Paraná,Índia,Agosto,2025,1.0,08,2025-08-31,2025-08-31,Região Sul,APAC
976741,Aérea,Distrito Federal,Índia,Agosto,2025,11.0,08,2025-08-31,2025-08-31,Região Centro-Oeste,APAC
976742,Terrestre,Paraná,Índia,Agosto,2025,1.0,08,2025-08-31,2025-08-31,Região Sul,APAC
976743,Terrestre,Paraná,Índia,Agosto,2025,31.0,08,2025-08-31,2025-08-31,Região Sul,APAC
976744,Terrestre,Rio Grande do Sul,Índia,Agosto,2025,2.0,08,2025-08-31,2025-08-31,Região Sul,APAC


In [6]:
test_df = df[df['date'] >= test_start]
assert test_df['date'].nunique() == TEST_MONTHS
print(
    f'Período do conjunto de teste: {test_df["date"].min():%d %B %Y} a '
    f'{test_df["date"].max():%d %B %Y}'
)
test_df.tail()


Período do conjunto de teste: 30 September 2025 a 31 August 2026


,access_route,state,origin_country,month,year,arrival_count,month_num,date,year_month,state_region,country_cultural_subdivision
985269,Terrestre,Rio Grande do Sul,Índia,Dezembro,2025,5.0,12,2025-12-31,2025-12-31,Região Sul,APAC
985270,Marítima,Alagoas,Índia,Dezembro,2025,3.0,12,2025-12-31,2025-12-31,Região Nordeste,APAC
985271,Marítima,Santa Catarina,Índia,Dezembro,2025,3.0,12,2025-12-31,2025-12-31,Região Sul,APAC
985272,Marítima,Santa Catarina,Índia,Dezembro,2025,5.0,12,2025-12-31,2025-12-31,Região Sul,APAC
985273,Fluvial,Amapá,Índia,Dezembro,2025,2.0,12,2025-12-31,2025-12-31,Região Norte,APAC


## Saída


In [7]:
# os três blocos devem ser consecutivos e cobrir cada linha exatamente uma vez
assert len(train_df) + len(val_df) + len(test_df) == len(df)
assert val_df['date'].min() == train_df['date'].max() + pd.offsets.MonthEnd(1)
assert test_df['date'].min() == val_df['date'].max() + pd.offsets.MonthEnd(1)

train_df.to_parquet(os.path.join(DATA_FOLDER_PATH, 'train_ptbr.parquet'), index=False)
val_df.to_parquet(os.path.join(DATA_FOLDER_PATH, 'val_ptbr.parquet'), index=False)
test_df.to_parquet(os.path.join(DATA_FOLDER_PATH, 'test_ptbr.parquet'), index=False)
